# Prepare AlphaGenome fold BEDs

Reproduces [this Colab](https://colab.research.google.com/drive/1qjR2W1LpRbrb0zmGescaiUGe7Ft1k0kN)
using **ENCFF802KJC / ENCSR200OML (IMR-90)**, and repeats the filter for AG FOLD_0 and FOLD_1.
Both start with **ChromBPNet fold_0 test peaks**, as in the Colab.
The original Colab uses ENCFF574LOE (another experiment). We use ENCFF802KJC
to reproduce the BEDs in our colleague's repository.

Run all cells with the BenchAG `.venv` kernel. Install missing dependencies once:
`pip install alphagenome==0.7.0 pyranges==0.1.4 pandas requests`.
No API key or predictions are needed.


In [1]:
from pathlib import Path
import gzip
import hashlib
import json
import tarfile

import pandas as pd
import pyranges as pr
import requests
from alphagenome.models.dna_client import ModelVersion, Organism
from alphagenome.data import fold_intervals

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'benchmarks/atac').is_dir())
DATA = ROOT / 'data/atac'
DATA.mkdir(parents=True, exist_ok=True)


/media/eternus1/nfs/projects/dpanchenko/BenchAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download the original ChromBPNet peaks

[ENCODE archive ENCFF802KJC](https://www.encodeproject.org/files/ENCFF802KJC/).
The download is reused if already present and checked against ENCODE's MD5.


In [2]:
accession = 'ENCFF802KJC'
url = f'https://www.encodeproject.org/files/{accession}/@@download/'
archive = DATA / f'{accession}.tar.gz'
expected_md5 = '40bbb77a35f5b35cc5fd13fbfa78ee7f'  # ENCODE checksum
if not archive.exists():
    temporary = archive.with_suffix('.part')
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with temporary.open('wb') as output:
            for block in response.iter_content(1024 * 1024):
                output.write(block)
    temporary.replace(archive)
checksum = hashlib.md5(archive.read_bytes()).hexdigest()
assert checksum == expected_md5, 'Archive checksum mismatch'

columns = ['Chromosome','Start','End','Name','Score','Strand','Signal','pValue','qValue','Summit']
member = 'fold_0/peaks.testset.fold_0.ENCSR200OML.bed.gz'
with tarfile.open(archive) as tar:
    entry = next(m for m in tar.getmembers() if m.name.lstrip('./') == member)
    with tar.extractfile(entry) as compressed:
        df_test = pd.read_csv(compressed, compression='gzip', sep='\t', header=None, names=columns)
assert len(df_test) == 59007
print(f'Original ChromBPNet fold_0 test peaks: {len(df_test):,}')
display(df_test.head())


Original ChromBPNet fold_0 test peaks: 59,007


,Chromosome,Start,End,Name,Score,Strand,Signal,pValue,qValue,Summit
0,chr1,100009473,100009794,Peak_115650,326,.,7.94722,32.62753,30.51946,155
1,chr1,100037591,100039086,Peak_249696,79,.,2.00528,7.92795,6.20955,207
2,chr1,100037591,100039086,Peak_40031,1000,.,6.54660,115.54169,112.90400,784
3,chr1,100037591,100039086,Peak_42084,1000,.,6.38382,110.52818,107.91309,962
4,chr1,100037591,100039086,Peak_4969,1000,.,16.62276,521.36438,517.69739,494


## 2. Remove AlphaGenome training overlaps

Exactly as in the Colab: load AG training intervals, resize them to **2²⁰ bp**,
and remove each peak that overlaps any training interval. Repeat for both models.
The bundled Borzoi interval file is the same partition source used by the AG API.

The final BED lines are sorted lexicographically to match the colleague's file byte for byte. This changes only the row order.


In [3]:
partition_file = ROOT / 'benchmarks/atac/data/sequences_human.bed.gz'
summary = []
for version, filename in [(ModelVersion.FOLD_0, 'fold0_test_regions.bed'),
                          (ModelVersion.FOLD_1, 'fold1_test_regions.bed')]:
    df_train = fold_intervals.get_fold_intervals(
        model_version=version, organism=Organism.HOMO_SAPIENS,
        subset=fold_intervals.Subset.TRAIN,
        example_regions_path=str(partition_file),
    ).rename(columns={'chromosome':'Chromosome', 'start':'Start', 'end':'End'}).copy()
    center = (df_train.Start + df_train.End) // 2
    df_train['Start'] = center - 2**19
    df_train['End'] = df_train['Start'] + 2**20

    gr_train = pr.PyRanges(df_train)
    filtered = pr.PyRanges(df_test).overlap(gr_train, invert=True)
    assert filtered.overlap(gr_train).empty, 'Some peaks overlap training intervals'
    df_filtered = filtered.df[columns].sort_values(['Chromosome','Start'])
    output = DATA / filename
    # The colleague's files are sorted lexicographically by the complete BED line.
    bed_text = df_filtered.to_csv(sep='\t', header=False, index=False)
    output.write_text(''.join(sorted(bed_text.splitlines(keepends=True))))
    summary.append({'model':version.name, 'original_peaks':len(df_test),
                    'retained_peaks':len(df_filtered), 'output':str(output.relative_to(ROOT)),
                    'sha256':hashlib.sha256(output.read_bytes()).hexdigest()})
assert [row['retained_peaks'] for row in summary] == [11860, 13812]
# Check every record, including summit and signal columns, against the colleague's BED.
reference = pd.read_csv(ROOT / 'benchmarks/atac/data/fold1_test_regions.bed', sep='\t', header=None, names=columns)
rebuilt = pd.read_csv(DATA / 'fold1_test_regions.bed', sep='\t', header=None, names=columns)
pd.testing.assert_frame_equal(rebuilt, reference, check_exact=False, rtol=1e-12, atol=1e-12)
assert (DATA / 'fold1_test_regions.bed').read_bytes() == (ROOT / 'benchmarks/atac/data/fold1_test_regions.bed').read_bytes()
print("PASS: FOLD_1 matches the colleague's BED.")
display(pd.DataFrame(summary)[['model','original_peaks','retained_peaks','output']])


PASS: FOLD_1 matches the colleague's BED.


,model,original_peaks,retained_peaks,output
0,FOLD_0,59007,11860,data/atac/fold0_test_regions.bed
1,FOLD_1,59007,13812,data/atac/fold1_test_regions.bed


## 3. Save the source and checks

Files are saved under `data/atac/`: `fold0_test_regions.bed` and
`fold1_test_regions.bed`. No model predictions are run.


In [4]:
manifest = {
    'colab':'https://colab.research.google.com/drive/1qjR2W1LpRbrb0zmGescaiUGe7Ft1k0kN',
    'archive_url':url, 'archive_md5':checksum, 'experiment':'ENCSR200OML',
    'peak_member':member, 'training_interval_length':2**20,
    'partition_sha256':hashlib.sha256(partition_file.read_bytes()).hexdigest(),
    'filter':'PyRanges.overlap(training_intervals, invert=True)',
    'outputs':summary,
}
(DATA / 'bed_preparation.json').write_text(json.dumps(manifest, indent=2) + '\n')
print('Saved:')
for row in summary:
    print(DATA / Path(row['output']).name)


Saved:
/media/eternus1/nfs/projects/dpanchenko/BenchAG/data/atac/fold0_test_regions.bed
/media/eternus1/nfs/projects/dpanchenko/BenchAG/data/atac/fold1_test_regions.bed
